### objective 4

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error
import joblib
import os

os.makedirs("../models", exist_ok=True)
os.makedirs("../reports/figures", exist_ok=True)

In [2]:
df = pd.read_csv("../data/processed/punjab_dataset.csv")

features = [c for c in df.columns if "_JJAS" in c and "PUNJAB" not in c]
X = df[features]
y = df["PUNJAB_JJAS"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [3]:
models = {
    "DecisionTree": DecisionTreeRegressor(max_depth=5, random_state=42),
    "RandomForest": RandomForestRegressor(n_estimators=100, random_state=42),
}

In [4]:
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    r2 = r2_score(y_test, pred)
    mae = mean_absolute_error(y_test, pred)
    print(name, "R2:", round(r2, 4), "MAE:", round(mae, 2))
    results.append([name, r2, mae])
    joblib.dump(model, f"../models/obj4_{name.lower()}.pkl")
    
pd.DataFrame(results, columns=["Model", "R2", "MAE"]).to_csv(
    "../models/obj4_comparison.csv", index=False)

DecisionTree R2: -0.222 MAE: 103.79
RandomForest R2: 0.5723 MAE: 70.44


In [5]:
rf = models["RandomForest"]
importance = pd.Series(rf.feature_importances_, index=features).sort_values()
importance.plot(kind="barh", title="Important Neighbours for Punjab")
plt.tight_layout()
plt.savefig("../reports/figures/obj4_feature_importance.png")
plt.close()